# 第15章：因子模型与归因

用固定种子20261015生成从2010年1月开始的120个月、五列因子和一个资产，设RF=0.001。在这组构造数据上拟合CAPM、FF3与FF5，比较HAC带宽、方差分解和归因，并将系数CSV、摘要JSON及三幅图写入RUN_DIR。这里FF3和FF5使用完全相同的共同因子列；历史French快照及项目九模型报告有各自的身份，不能混作本次结果。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch15_factor_models_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

import pandas as pd


from quant_course.factor_models import fit_monthly_factor_model
from ch15_factor_models import (
    build_summary,
    fit_models,
    load_sample,
    make_figures,
    write_artifacts,
)
from IPython.display import display


## 1. 构造样本与来源

代码导入历史加载器但不调用它，实际单元生成一个资产与共享的五列因子，不下载市场数据、不核验历史快照，也不填补缺月。NoDur属于另行保留的French历史行业说明，不是当前构造资产。

In [ ]:
import numpy as np
rng = np.random.default_rng(20261015)
idx = pd.period_range("2010-01", periods=120, freq="M")
factors = pd.DataFrame(rng.normal(0, .025, (120, 5)), index=idx, columns=["Mkt-RF", "SMB", "HML", "RMW", "CMA"])
factors["RF"] = .001
frames = {"ff3": factors[["Mkt-RF", "SMB", "HML", "RF"]].copy(), "ff5": factors.copy()}
total = pd.Series(.001 + .002 + factors["Mkt-RF"] + .3*factors["HML"] + rng.normal(0, .01, 120), index=idx, name="CONSTRUCTED asset")
print("CONSTRUCTED seeded factor-regression inputs; French branch skipped")
assert len(total) == 120 and frames["ff5"].index.equals(total.index)


收益与因子均为十进制月度数值，0.01表示月度1%。PeriodIndex标记月份区间，不是可得或执行时间。历史FF3和FF5发布快照的同名Mkt-RF、SMB不必相同；当前构造例则有意共用完全相同的列。

In [ ]:
display(total.head().to_frame())
display(frames["ff3"].head())
display(frames["ff5"].head())

## 2. 普通最小二乘与因子暴露

普通最小二乘（Ordinary Least Squares，OLS）估计资产超额收益对因子的月度线性关系。CAPM使用Mkt-RF；FF3加入小盘减大盘（Small Minus Big，SMB）和高账面市值比减低账面市值比（High Minus Low，HML）；FF5再加入盈利稳健减盈利弱（Robust Minus Weak，RMW）和投资保守减投资积极（Conservative Minus Aggressive，CMA）。资产总收益减匹配RF一次。

下格直接调用共享核心拟合CAPM，再由章节工作流拟合三个模型，比较两条路径的CAPM系数，不复制估计器。

In [ ]:
core_capm = fit_monthly_factor_model(
    total,
    frames["ff3"],
    factor_columns=("Mkt-RF",),
    hac_maxlags=6,
)
fits = fit_models(total, frames, hac_maxlags=6)

assert core_capm.coefficient_table.equals(fits["capm"].coefficient_table)
print("共享核心和章节工作流的 CAPM 系数表一致。")
print("模型：", list(fits))

## 3. 月度截距与不确定性

异方差与自相关稳健（Heteroskedasticity and Autocorrelation Consistent，HAC）标准误使用Bartlett核，带宽按月度观察行计。比较lag为3、6、12时的标准误和95%区间：截距点估计固定，估计不确定性会变。

t参考分布及有限样本修正给出依赖条件的近似推断，软件输出不能验证适用条件或证明alpha可交易。

In [ ]:
summary = build_summary(total, frames, fits, constructed=True)

alpha_rows = []
for name in ("capm", "ff3", "ff5"):
    model = summary["models"][name]
    for lag, values in model["alpha_hac_lag_sensitivity"].items():
        alpha_rows.append({
            "model": name,
            "HAC lags (months)": int(lag),
            "alpha (% per month)": values["estimate"] * 100,
            "HAC standard error (% per month)": values["standard_error_hac"] * 100,
            "95% CI low (% per month)": values["ci_low_hac"] * 100,
            "95% CI high (% per month)": values["ci_high_hac"] * 100,
        })

alpha_sensitivity = pd.DataFrame(alpha_rows)
display(alpha_sensitivity.pivot(
    index="model",
    columns="HAC lags (months)",
    values="alpha (% per month)",
))

CONSTRUCTED 输出由上方固定种子计算；原章节历史截距数字不属于本次计算。


In [ ]:
alpha_intervals = alpha_sensitivity[
    ["model", "HAC lags (months)", "HAC standard error (% per month)",
     "95% CI low (% per month)", "95% CI high (% per month)"]
]
display(alpha_intervals)

## 4. 样本方差分解

含截距的 OLS 在本样本内把资产超额收益写成拟合部分与残差之和。下面按样本方差分母 n−1 核对 Var(y) = Var(fitted) + Var(residual)。这是拟合样本的恒等关系，不代表未来残差独立，也不能据此预测下一期。

In [ ]:
variance_rows = []
for name in ("capm", "ff3", "ff5"):
    decomposition = summary["models"][name]["sample_variance_decomposition_ddof1"]
    variance_rows.append({
        "model": name,
        "observed excess variance, ddof=1": decomposition["observed_excess_return"],
        "fitted excess variance, ddof=1": decomposition["fitted_excess_return"],
        "residual variance, ddof=1": decomposition["residual"],
        "fitted + residual": decomposition["sum_fitted_plus_residual"],
        "identity verified": decomposition["identity_holds_within_tolerance"],
    })

display(pd.DataFrame(variance_rows).set_index("model"))

## 5. 构造FF5暴露与图形

载荷是对当前构造因子列的无量纲回归斜率，区间来自共享核心的HAC计算。Mkt-RF已经是超额收益，只有资产总收益一侧减RF。图形描述本次构造拟合，不是重新发布的French估计。

In [ ]:
display(fits["ff5"].coefficient_table)
print("R²:", fits["ff5"].r_squared)
print("Adjusted R²:", fits["ff5"].adjusted_r_squared)

贡献图逐月累加构造因子项、截距项和残差，再乘100表示百分点。它是算术归因，不是NAV或复利财富。beta乘因子值在这里只是回归项，不自动构成可投资的零美元组合。

In [ ]:
from IPython.display import Image, display

figure_info = make_figures(total, frames, fits, output_dir=RUN_DIR)
for key, metadata in figure_info.items():
    image_path = ROOT / metadata["path"]
    print(key, metadata["width_px"], "x", metadata["height_px"], "px")
    display(Image(filename=str(image_path)))

## 6. 输出与解释边界

最后一格将coefficients.csv和summary.json写入RUN_DIR，并保存本次构造图。当前FF3、FF5的共同列完全相同。

另属历史报告的French快照中，同名Mkt-RF、SMB曾存在数值差异且上游原因未确定，因此那些历史回归不严格嵌套，R²变化不能全归于RMW、CMA。此警示不能反过来描述当前共享列。历史修订快照不等于逐月PIT版本；这里没有逐股重建、成交、成本或净alpha证据。

In [ ]:
written = write_artifacts(fits, summary, coefficients_path=RUN_DIR / "coefficients.csv", summary_path=RUN_DIR / "summary.json")
print(written)
